In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"],capture_output=True,text=True).stdout)
import torch
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch.cuda.get_device_capability(0))
try:
    import triton; print("triton", triton.__version__)
except Exception as e: print("no triton", e)
import triton, triton.language as tl
@triton.jit
def k(x, y, o, M: tl.constexpr, N: tl.constexpr, K: tl.constexpr):
    a = tl.load(x + tl.arange(0,M)[:,None]*K + tl.arange(0,K)[None,:])
    b = tl.load(y + tl.arange(0,K)[:,None]*N + tl.arange(0,N)[None,:])
    tl.store(o + tl.arange(0,M)[:,None]*N + tl.arange(0,N)[None,:], tl.dot(a,b))
for dt in (torch.float16, torch.bfloat16):
    try:
        a=torch.randn(16,32,device="cuda",dtype=dt); b=torch.randn(32,16,device="cuda",dtype=dt); o=torch.empty(16,16,device="cuda",dtype=torch.float32)
        k[(1,)](a,b,o,16,16,32); torch.cuda.synchronize(); print(dt, "dot ok, err", (o-(a.float()@b.float())).abs().max().item())
    except Exception as e: print(dt, "dot FAIL", type(e).__name__, str(e)[:200])
